# Stage C4 — PINN Ensemble Training

Dual-fuel PINN pipeline | Sandrine Schueller Mafra | PPGEM – UFPR
Supports dissertation Sec. 3.5.2.

This is where every earlier C-stage lands: C1's constraints, C2's
composite loss, C3's tuned hyperparameters, trained for real —
two-phase (Adam, then L-BFGS) per member, 10 members for the ensemble
mean/uncertainty split described in Sec. 3.5.2.

**Framework note:** `tensorflow_probability` isn't installed here (no
network to add it), and Sec. 3.5.2's L-BFGS phase isn't something
Keras ships natively. This notebook bridges to `scipy.optimize.minimize
(method='L-BFGS-B')` instead — flatten the model's weights into one
vector, hand scipy a function that returns `(loss, gradient)` for any
vector it proposes, let it optimize, unflatten the result back into
the model. This is a standard technique for exactly this situation
(published PINN implementations use it when TFP isn't in play), but
it's still a bridge between two systems, not a single native call —
worth knowing if a shape or dtype error shows up here specifically.
The flatten/unflatten round-trip itself was checked against synthetic
weight tensors before this file was written; see chat.

**Runtime expectation:** 10 members x (Adam up to 5000 epochs + L-BFGS
200-500 iterations) is the heaviest stage yet. Expect it to run for a
while — Section 5 trains and inspects one member first specifically so
you're not waiting through all 10 before finding out something's wrong.

**Input:** `data/masters_data.xlsx`, `outputs/C1_collocation_points.csv`,
`outputs/C2_loss_config.json`, `outputs/C3_best_hyperparameters.json`
(falls back to C2's un-tuned config if C3 wasn't run — noted where it
matters below).
**Output:** 10 trained models, ensemble mean/std predictions,
`outputs/C4_ensemble_predictions.csv` — what **D1** compares against
the B2 baseline.


## Setup

In [ ]:
import json
import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from scipy.optimize import minimize
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("polars    ", pl.__version__)
import plotly
print("plotly    ", plotly.__version__)
print("tensorflow", tf.__version__)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "masters_data.xlsx"
OUT_DIR = PROJECT_ROOT / "outputs"
SEED = 42


## 1. Load data, collocation points, and upstream configs

In [ ]:
COLUMN_MAP = {
    "SOI [o.CA]": "SOI", "Lambda [-]": "lambda", "Sub. Rate [%]": "sub_rate",
    "Prail [bar]": "P_rail", "HC [g/kW.h]": "HC", "NOX [ppm]": "NOx",
    "CO2 [%]": "CO2", "SO_H [FSN]": "PM", "ETA [%]": "eta",
}
INPUT_COLS = ["SOI", "lambda", "sub_rate", "P_rail"]
OUTPUT_COLS = ["HC", "NOx", "CO2", "PM", "eta"]
ALL_COLS = INPUT_COLS + OUTPUT_COLS
N_IN, N_OUT = len(INPUT_COLS), len(OUTPUT_COLS)
SOI_IDX, LAMBDA_IDX, SUBRATE_IDX, PRAIL_IDX = [INPUT_COLS.index(c) for c in INPUT_COLS]
HC_IDX, NOX_IDX, CO2_IDX, PM_IDX, ETA_IDX = [OUTPUT_COLS.index(c) for c in OUTPUT_COLS]
EMISSION_IDXS = [HC_IDX, NOX_IDX, CO2_IDX, PM_IDX]

df = pl.read_excel(RAW_PATH).rename(COLUMN_MAP).select(ALL_COLS)
n = df.shape[0]
medians = {c: df[c].median() for c in INPUT_COLS}
ranges = {c: (df[c].max() - df[c].min()) for c in INPUT_COLS}
deviation = np.column_stack([np.abs(df[c].to_numpy() - medians[c]) / ranges[c] for c in INPUT_COLS])
raw_block = np.array(INPUT_COLS)[deviation.argmax(axis=1)]

def smooth_isolated_labels(labels, passes=2):
    out = list(labels)
    for _ in range(passes):
        changed = False
        for i in range(1, len(out) - 1):
            if out[i] != out[i - 1] and out[i - 1] == out[i + 1]:
                out[i] = out[i - 1]
                changed = True
        if not changed:
            break
    return np.array(out)

ofat_block = smooth_isolated_labels(raw_block)
extremity = np.zeros(n)
for b in np.unique(ofat_block):
    idx = np.where(ofat_block == b)[0]
    vals = df[b].to_numpy()[idx]
    order = np.argsort(vals)
    m = len(idx)
    pos = np.array([0.5]) if m == 1 else np.empty(m)
    if m > 1:
        ranks = np.empty(m)
        ranks[order] = np.arange(m)
        pos = ranks / (m - 1)
    extremity[idx] = np.abs(pos - 0.5) * 2
rng_np = np.random.default_rng(SEED)
jitter = rng_np.uniform(-1e-9, 1e-9, size=n)
order = np.argsort(-(extremity + jitter))
split = np.array(["train"] * n)
split[order[:6]] = "test"
split[order[6:12]] = "val"
df = df.with_columns(pl.Series("split", split))
train_df = df.filter(pl.col("split") == "train")
train_min = {c: train_df[c].min() for c in ALL_COLS}
train_max = {c: train_df[c].max() for c in ALL_COLS}
df = df.with_columns([
    ((pl.col(c) - train_min[c]) / (train_max[c] - train_min[c])).alias(f"{c}_norm")
    for c in ALL_COLS
])

def split_arrays(split_name, cols):
    sub = df.filter(pl.col("split") == split_name)
    return sub.select([f"{c}_norm" for c in cols]).to_numpy().astype(np.float32)

X_train, Y_train = split_arrays("train", INPUT_COLS), split_arrays("train", OUTPUT_COLS)
X_val, Y_val = split_arrays("val", INPUT_COLS), split_arrays("val", OUTPUT_COLS)
X_test, Y_test = split_arrays("test", INPUT_COLS), split_arrays("test", OUTPUT_COLS)
sigma2 = np.maximum(Y_train.var(axis=0, ddof=1), 1e-8)

colloc_df = pl.read_csv(OUT_DIR / "C1_collocation_points.csv")
X_colloc = colloc_df.select(INPUT_COLS).to_numpy().astype(np.float32)
rho_colloc = colloc_df["density_ratio"].to_numpy()
validity_eff_nox = colloc_df["validity_eff_nox"].to_numpy()

with open(OUT_DIR / "C2_loss_config.json") as f:
    c2_config = json.load(f)
w_final_base = c2_config["w_final"]

c3_path = OUT_DIR / "C3_best_hyperparameters.json"
if c3_path.exists():
    with open(c3_path) as f:
        c3 = json.load(f)
    hp = c3["best_hp"]
    print(f"Loaded C3-tuned hyperparameters (used_optuna={c3.get('used_optuna')})")
else:
    hp = {"n_hidden_layers": 1, "neurons_per_layer": 64, "activation": "tanh",
          "learning_rate": 1e-3, "batch_size": 16, "beta1": 0.9, "beta2": 0.999,
          "monotonic_scale": 1.0, "shape_scale": 1.0, "tradeoff_scale": 1.0,
          "nonneg_scale": 1.0, "weight_decay": 1e-5, "dropout_rate": 0.0}
    print("WARNING: C3 output not found -- using an untuned fallback config. Run C3 first.")

scaled_w = {
    "NOx-SOI": w_final_base["NOx-SOI"] * hp["monotonic_scale"],
    "PM-lambda": w_final_base["PM-lambda"] * hp["monotonic_scale"],
    "HC-lambda": w_final_base["HC-lambda"] * hp["shape_scale"],
    "NOx-PM": w_final_base["NOx-PM"] * hp["tradeoff_scale"],
    "eta-NOx": w_final_base["eta-NOx"] * hp["tradeoff_scale"],
    "non-negativity": w_final_base["non-negativity"] * hp["nonneg_scale"],
}
K_SCHEDULE = c2_config.get("k_schedule", 0.01)
T0_SCHEDULE = c2_config.get("t0_schedule", 750)
print("scaled physics weights:", {k: round(v, 3) for k, v in scaled_w.items()})


## 2. Loss machinery (same formulas as C1/C2/C3, tuned config applied)

In [ ]:
def build_pinn(seed):
    tf.random.set_seed(seed)
    model = keras.Sequential([keras.Input(shape=(N_IN,))])
    for _ in range(hp["n_hidden_layers"]):
        model.add(layers.Dense(hp["neurons_per_layer"], activation=hp["activation"]))
        if hp["dropout_rate"] > 0:
            model.add(layers.Dropout(hp["dropout_rate"]))
    model.add(layers.Dense(N_OUT, activation="linear"))
    return model

def monotonic_decreasing_per_point(predict_fn, x, out_idx, in_idx):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x_t)
        target = predict_fn(x_t)[:, out_idx]
    d = tape.gradient(target, x_t)[:, in_idx]
    return tf.square(tf.maximum(0.0, d))

def convexity_per_point(predict_fn, x, out_idx, in_idx):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape() as tape2:
        tape2.watch(x_t)
        with tf.GradientTape() as tape1:
            tape1.watch(x_t)
            target = predict_fn(x_t)[:, out_idx]
        d_first = tape1.gradient(target, x_t)[:, in_idx]
    d_second = tape2.gradient(d_first, x_t)[:, in_idx]
    return tf.square(tf.maximum(0.0, -d_second))

def tradeoff_per_point(predict_fn, x, out_idx_a, out_idx_b, in_idx):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape(persistent=True) as tape:
        tape.watch(x_t)
        y = predict_fn(x_t)
        a, b = y[:, out_idx_a], y[:, out_idx_b]
    grad_a = tape.gradient(a, x_t)[:, in_idx]
    grad_b = tape.gradient(b, x_t)[:, in_idx]
    del tape
    return tf.square(tf.maximum(0.0, grad_a * grad_b))

def nonneg_loss(predict_fn, x, out_idxs=EMISSION_IDXS):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    y = predict_fn(x_t)
    emissions = tf.gather(y, out_idxs, axis=1)
    return tf.reduce_mean(tf.reduce_sum(tf.square(tf.maximum(0.0, -emissions)), axis=1))

def weighted_physics_loss(per_point, lambda_x):
    return tf.reduce_mean(tf.constant(lambda_x, dtype=tf.float32) * per_point)

def data_loss(predict_fn, X, Y):
    pred = predict_fn(tf.convert_to_tensor(X, dtype=tf.float32))
    sq_err = tf.square(pred - tf.constant(Y)) / tf.constant(sigma2, dtype=tf.float32)
    return tf.reduce_mean(tf.reduce_sum(sq_err, axis=1))

def regularization_loss(model):
    P = model.count_params()
    sq_sum = tf.add_n([tf.reduce_sum(tf.square(w)) for w in model.trainable_weights if len(w.shape) > 1])
    return sq_sum / P

def scheduled_weight(t, w_j_final, k=K_SCHEDULE, t0=T0_SCHEDULE):
    return w_j_final / (1 + np.exp(-k * (t - t0)))

def full_loss(model, X, Y, epoch, training=True):
    predict_fn = lambda x: model(x, training=training)
    l_data = data_loss(predict_fn, X, Y)
    l_phys = 0.0
    for name, fn in [
        ("NOx-SOI", lambda: monotonic_decreasing_per_point(predict_fn, X_colloc, NOX_IDX, SOI_IDX)),
        ("PM-lambda", lambda: monotonic_decreasing_per_point(predict_fn, X_colloc, PM_IDX, LAMBDA_IDX)),
        ("HC-lambda", lambda: convexity_per_point(predict_fn, X_colloc, HC_IDX, LAMBDA_IDX)),
        ("NOx-PM", lambda: tradeoff_per_point(predict_fn, X_colloc, NOX_IDX, PM_IDX, SOI_IDX)),
        ("eta-NOx", lambda: tradeoff_per_point(predict_fn, X_colloc, ETA_IDX, NOX_IDX, SOI_IDX)),
    ]:
        validity = validity_eff_nox if name == "eta-NOx" else np.ones(len(X_colloc))
        lambda_x = rho_colloc * validity
        l_phys = l_phys + float(scheduled_weight(epoch, scaled_w[name])) * weighted_physics_loss(fn(), lambda_x)
    l_phys = l_phys + float(scheduled_weight(epoch, scaled_w["non-negativity"])) * nonneg_loss(predict_fn, X_colloc)
    l_reg = regularization_loss(model)
    return l_data + l_phys + hp["weight_decay"] * l_reg


## 3. Phase 1 — Adam (Sec. 3.5.2)

Up to 5000 epochs, early stopping at 200 stagnant epochs, learning
rate x 0.5 after 50 stagnant epochs — the full budget this time (C3
used a reduced version to keep 50 x 3-fold search trials tractable;
there's only one training run per ensemble member here).

In [ ]:
def train_adam_phase(model, seed, max_epochs=5000, patience=200, lr_patience=50, verbose_every=250):
    optimizer = keras.optimizers.Adam(learning_rate=hp["learning_rate"],
                                       beta_1=hp["beta1"], beta_2=hp["beta2"])
    history = {"epoch": [], "train_loss": [], "val_loss": []}
    best_val, best_weights, no_improve, plateau = np.inf, None, 0, 0

    for epoch in range(max_epochs):
        idx = np.random.default_rng(seed + epoch).permutation(len(X_train))
        for start in range(0, len(idx), hp["batch_size"]):
            batch = idx[start:start + hp["batch_size"]]
            with tf.GradientTape() as tape:
                loss = full_loss(model, X_train[batch], Y_train[batch], epoch, training=True)
            grads = tape.gradient(loss, model.trainable_weights)
            optimizer.apply_gradients(zip(grads, model.trainable_weights))

        train_loss = float(full_loss(model, X_train, Y_train, epoch, training=False))
        val_loss = float(data_loss(lambda x: model(x, training=False), X_val, Y_val))
        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if val_loss < best_val - 1e-7:
            best_val, best_weights, no_improve, plateau = val_loss, [w.numpy().copy() for w in model.trainable_weights], 0, 0
        else:
            no_improve += 1
            plateau += 1
        if plateau >= lr_patience:
            optimizer.learning_rate.assign(optimizer.learning_rate * 0.5)
            plateau = 0
        if epoch % verbose_every == 0:
            print(f"    epoch {epoch:5d}  train_loss={train_loss:.5f}  val_loss={val_loss:.5f}  lr={float(optimizer.learning_rate):.2e}")
        if no_improve >= patience:
            print(f"    early stop at epoch {epoch} (no val improvement for {patience} epochs)")
            break

    if best_weights is not None:
        for w, bw in zip(model.trainable_weights, best_weights):
            w.assign(bw)
    return history


## 4. Phase 2 — L-BFGS (Sec. 3.5.2)

The scipy bridge: flatten the model's weights into one vector,
minimize `full_loss` as a function of that vector (returning both loss
and gradient — `jac=True` tells scipy the objective already provides
the gradient instead of estimating it by finite differences), unflatten
the result back onto the model.

In [ ]:
def flatten_weights(model):
    return np.concatenate([w.numpy().ravel() for w in model.trainable_weights]).astype(np.float64)

def unflatten_and_assign(model, flat_vec):
    idx = 0
    for w in model.trainable_weights:
        size = int(np.prod(w.shape))
        w.assign(flat_vec[idx:idx + size].reshape(w.shape).astype(np.float32))
        idx += size

def train_lbfgs_phase(model, final_epoch, maxiter=500):
    def objective(flat_vec):
        unflatten_and_assign(model, flat_vec)
        with tf.GradientTape() as tape:
            loss = full_loss(model, X_train, Y_train, final_epoch, training=True)
        grads = tape.gradient(loss, model.trainable_weights)
        flat_grads = np.concatenate([g.numpy().ravel() for g in grads]).astype(np.float64)
        return float(loss.numpy()), flat_grads

    x0 = flatten_weights(model)
    result = minimize(objective, x0, jac=True, method="L-BFGS-B",
                       options={"maxiter": maxiter, "maxfun": maxiter * 2})
    unflatten_and_assign(model, result.x)
    return result


## 5. One ensemble member, end to end — sanity check before running all 10

Confirms L-BFGS actually improves on where Adam left off, which is
what Sec. 3.5.2 claims it should do — worth seeing on one member before
committing to all 10.

In [ ]:
tf.random.set_seed(0)
member0 = build_pinn(seed=0)

print("Phase 1 (Adam):")
hist0 = train_adam_phase(member0, seed=0)
loss_after_adam = hist0["train_loss"][-1]
final_epoch = hist0["epoch"][-1]

print("\nPhase 2 (L-BFGS):")
result0 = train_lbfgs_phase(member0, final_epoch=final_epoch)
loss_after_lbfgs = float(full_loss(member0, X_train, Y_train, final_epoch, training=False))

print(f"\nloss after Adam:   {loss_after_adam:.6f}")
print(f"loss after L-BFGS: {loss_after_lbfgs:.6f}  "
      f"({'improved' if loss_after_lbfgs < loss_after_adam else 'DID NOT improve -- check maxiter / convergence'})")
print(f"scipy L-BFGS-B converged: {result0.success}, iterations: {result0.nit}")


## 6. Train the full ensemble of 10

In [ ]:
N_ENSEMBLE = 10
ensemble_models = []
ensemble_histories = []

for m in range(N_ENSEMBLE):
    print(f"\n=== member {m+1}/{N_ENSEMBLE} (seed={m}) ===")
    keras.backend.clear_session()
    model = build_pinn(seed=m)
    hist = train_adam_phase(model, seed=m, verbose_every=1000)
    final_epoch = hist["epoch"][-1]
    result = train_lbfgs_phase(model, final_epoch=final_epoch)
    hist["lbfgs_final_loss"] = float(full_loss(model, X_train, Y_train, final_epoch, training=False))
    hist["lbfgs_converged"] = bool(result.success)
    ensemble_models.append(model)
    ensemble_histories.append(hist)

print(f"\nAll {N_ENSEMBLE} members trained.")


## 7. Training curves, all 10 members (Adam phase; L-BFGS is a single post-hoc jump, marked separately)

In [ ]:
fig = go.Figure()
palette = ["#185FA5", "#993C1D", "#534AB7", "#3B6D11", "#854F0B",
           "#5A5A55", "#1D9E75", "#B23A6B", "#3D5A80", "#7A5C3E"]
for m, (hist, color) in enumerate(zip(ensemble_histories, palette)):
    fig.add_trace(go.Scatter(x=hist["epoch"], y=hist["train_loss"], mode="lines",
                              line=dict(color=color, width=1.3), name=f"member {m}", opacity=0.8))
    fig.add_trace(go.Scatter(x=[hist["epoch"][-1]], y=[hist["lbfgs_final_loss"]], mode="markers",
                              marker=dict(color=color, size=10, symbol="star"), showlegend=False))
fig.update_layout(title="Adam training curves (lines) + L-BFGS final loss (stars), all 10 members",
                   xaxis_title="epoch", yaxis_title="total loss", yaxis_type="log", width=900, height=500)
fig.show()


## 8. Ensemble predictions — mean and uncertainty

$$
\hat y(x) = \frac{1}{10}\sum_{m=1}^{10} \hat y_m(x), \qquad
\sigma_{\text{ens}}(x) = \sqrt{\frac{1}{10}\sum_{m=1}^{10}\left(\hat y_m(x) - \hat y(x)\right)^2}
$$

In [ ]:
def ensemble_predict(X):
    preds = np.stack([m(X, training=False).numpy() for m in ensemble_models], axis=0)  # [10, N, 5]
    return preds.mean(axis=0), preds.std(axis=0), preds

mean_train, std_train, all_train = ensemble_predict(X_train)
mean_val, std_val, all_val = ensemble_predict(X_val)
mean_test, std_test, all_test = ensemble_predict(X_test)
print("mean/std shapes:", mean_test.shape, std_test.shape)
print("mean test-set ensemble std by output:", dict(zip(OUTPUT_COLS, std_test.mean(axis=0).round(4))))


## 9. Diversity check — did the 10 members actually find different solutions?

If they collapsed to (near-)identical weights, the ensemble std above
is meaningless as an uncertainty estimate — it would just be numerical
noise, not genuine disagreement between local minima.

In [ ]:
spread_per_output = std_test.mean(axis=0)
fig = make_subplots(rows=1, cols=2, subplot_titles=["Ensemble std by output (test)", "Prediction spread, HC (test points)"])
fig.add_trace(go.Bar(x=OUTPUT_COLS, y=spread_per_output, marker_color="#534AB7", showlegend=False), row=1, col=1)

for i in range(all_test.shape[1]):
    fig.add_trace(go.Box(y=all_test[:, i, HC_IDX], name=f"pt {i}", marker_color="#185FA5", showlegend=False),
                  row=1, col=2)
fig.update_layout(height=420, width=900, title_text="Ensemble diversity")
fig.show()

near_zero_std = (spread_per_output < 1e-4).sum()
print(f"outputs with near-zero ensemble std (<1e-4): {near_zero_std} of {N_OUT} "
      f"({'members may have collapsed to the same solution -- check seeds/init' if near_zero_std > 0 else 'OK, members disagree meaningfully'})")


## Optional — persist outputs

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

pred_rows = []
for s, (X_s, mean_s, std_s) in {
    "train": (X_train, mean_train, std_train),
    "val": (X_val, mean_val, std_val),
    "test": (X_test, mean_test, std_test),
}.items():
    for j in range(X_s.shape[0]):
        row = {"split": s}
        for i, out in enumerate(OUTPUT_COLS):
            row[f"{out}_mean_norm"] = float(mean_s[j, i])
            row[f"{out}_std_norm"] = float(std_s[j, i])
            row[f"{out}_mean"] = float(mean_s[j, i] * (train_max[out] - train_min[out]) + train_min[out])
        pred_rows.append(row)
pl.DataFrame(pred_rows).write_csv(OUT_DIR / "C4_ensemble_predictions.csv")

for m, model in enumerate(ensemble_models):
    try:
        model.save(OUT_DIR / f"C4_ensemble_member_{m}.keras")
    except Exception as e:
        print(f"member {m}: '.keras' save failed ({e}), falling back to '.h5'")
        model.save(OUT_DIR / f"C4_ensemble_member_{m}.h5")

print(f"Saved to {OUT_DIR}")


## Next

**D1** loads `C4_ensemble_predictions.csv` alongside B2's baseline
predictions and runs the actual PINN-vs-baseline comparison (Eq.
3.21–3.27) that everything since Phase A has been building toward.
